In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from google.colab import drive
drive.mount('/content/drive')

df0 = pd.read_excel('/content/drive/MyDrive/80-drivers_data.xlsx')

Mounted at /content/drive


In [ ]:
import random
import numpy as np
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from torch.nn.utils.rnn import pad_sequence

seq_len = 10
batch_size = 32


random.seed(42)
scaler = StandardScaler()
feature_columns = ['Accelerator pedal position', 'Current gear shift position', 'slope']
other_columns = ['driver_name']
df0 = df0[feature_columns + other_columns]
df0[feature_columns] = scaler.fit_transform(df0[feature_columns])
data = []

for driver, df in df0.groupby('driver_name'):
    series = df[feature_columns].values.astype(np.float32)
    l = series.shape[0] - (series.shape[0] % seq_len ) - seq_len
    for i in range(l):
      data.append(series[i:i+seq_len])
sequences = np.array(data)
print(sequences.shape[0], sequences.dtype)
train_loader = DataLoader(sequences, batch_size=batch_size, shuffle=False)


55450 float32


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Hyperparameters
input_dim = noise_dim = 10  # Input to the generator
hidden_dim = 64
output_dim = sequences.shape[2]  # Number of features in your dataset
num_layers = 2
num_epochs = 50
batch_size = 64
learning_rate = 0.0002

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

class LSTMGenerator(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers):
        super(LSTMGenerator, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out)
        return out

class LSTMDiscriminator(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers):
        super(LSTMDiscriminator, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)

        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return self.sigmoid(out)


# Initialize models
generator = LSTMGenerator(noise_dim, hidden_dim, output_dim, num_layers).to(device)
discriminator = LSTMDiscriminator(output_dim, hidden_dim, num_layers).to(device)

# Loss function and optimizers
criterion = nn.BCELoss()
optimizer_g = optim.Adam(generator.parameters(), lr=learning_rate)
optimizer_d = optim.Adam(discriminator.parameters(), lr=learning_rate)

cuda


In [ ]:


# Training Loop
for epoch in range(num_epochs):
    for real_data in train_loader:
        real_data = real_data.to(device)
        batch_size = real_data.size(0)

        # Train Discriminator
        optimizer_d.zero_grad()
        optimizer_g.zero_grad()
        noise = torch.randn(batch_size, seq_len, noise_dim).to(device)
        fake_data = generator(noise)

        real_labels = torch.ones(batch_size, 1).to(device)
        fake_labels = torch.zeros(batch_size, 1).to(device)

        real_loss = criterion(discriminator(real_data), real_labels)
        fake_loss = criterion(discriminator(fake_data.detach()), fake_labels)
        d_loss = real_loss + fake_loss
        d_loss.backward()
        optimizer_d.step()

        # Train Generator
        optimizer_d.zero_grad()
        optimizer_g.zero_grad()
        fake_labels = torch.ones(batch_size, 1).to(device)  # Generator wants to fool the discriminator
        g_loss = criterion(discriminator(fake_data), fake_labels)
        g_loss.backward()
        optimizer_g.step()

    print(f'Epoch [{epoch}/{num_epochs}], d_loss: {d_loss.item():.4f}, g_loss: {g_loss.item():.4f}')


In [ ]:
torch.save(generator.state_dict(), "LSTMGenerator.pth")
torch.save(discriminator.state_dict(), "LSTMDiscriminator.pth")

In [ ]:
generator.eval()
with torch.no_grad():
  noise = torch.randn(batch_size, seq_len, noise_dim).to(device)
  fake_data = generator(noise).cpu().numpy()

  b = scaler.inverse_transform(fake_data[0])
  print(b.tolist())